<a href="https://colab.research.google.com/github/anisrabiu/PINNs/blob/master/22bb%2B_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#code ref:
#CursorPINN [train_pinn_gothic_Temp5_enhanced]
#a PyTorch PINN code to solve the 2-D Heat Diffusion, Transient Heat-equations in a greenhouse of size 6.5 m × 4.5 m.
#
#20/10/2025

#Improve predict temperature with implement greenhouse control logic
#Side vent 23C
#Roof vent 21C
#heating    8C
#Corrected version of examples 22a By using Gimini edit
#Example22b+ (newly developed wen 22b lost)
#hourly data for 7-days 21-27 Feb 2023

In [ ]:
import os
import math
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from typing import Tuple, Dict


In [ ]:
# ---------------------------
# Configuration
# ---------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32

torch.set_default_dtype(DTYPE)

# Geometry
W = 6.5
H = 4.5
xc = 3.25
R = 3.565
yc = 0.435
# ridge parameters
sv = 0.56  # half ridge width (m) as provided
apex_x = xc
apex_y = 4.5
roof_vent_half_width = 0.45  # 0.9 m total

# Physics (heat diffusion)
alpha = 2.1e-5  # m^2/s (thermal diffusivity of air)

# Time - based on actual data duration
T_final = 7 * 24 * 3600.0  # 7 days * 24 hours/day * 3600 seconds/hour
# T_final = 13800.0  # 23 hours (13800 seconds) based on CSV data

# Network/training
HID = 64
LAYERS = 6
LR = 1e-3
EPOCHS = 10000
SAVE_EVERY = 2000
BATCH_F_INNER = 10000
BATCH_BC = 2000
BATCH_INIT = 4000
BATCH_DATA = 256

# OUTPUT_DIR = os.path.join(os.path.dirname(__file__), 'outputs_gothic_temp_enhanced')
OUTPUT_DIR = 'outputs_gothic_temp_enhanced' # Removed os.path.dirname(__file__)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load day data CSV file
# DAY_DATA_PATH = os.path.join(os.path.dirname(__file__), '20230227 temperature data.csv')
DAY_DATA_PATH = '/content/202302 21-27 temperature data hourly.csv' # Corrected path based on available files
DAY_DATA_PATH = os.path.abspath(DAY_DATA_PATH)

# Add characteristic scales for normalization
Lc = W
Hc = H
Tc = T_final  # Use actual data duration for time normalization
Theta_c = 10.0  # characteristic temperature scale (degC) used to non-dimensionalize outputs

# Greenhouse control parameters
SIDE_VENT_TEMP = 23.0  # °C - side vents open when inside temp reaches this
ROOF_VENT_TEMP = 21.0  # °C - roof vents open when inside temp reaches this
HEATING_TEMP = 8.0     # °C - heating starts when inside temp drops to this
HEATING_POWER = 50.0   # W/m³ - heating power density

In [ ]:
# ---------------------------
# Utility: roof curve and helpers
# ---------------------------

def roof_arc_y(x: torch.Tensor) -> torch.Tensor:
	"""Circular arc y for |x-xc| >= sv region.
	Pass torch tensor x.
	"""
	dx = x - xc
	# ensure inside circle domain
	val = yc + torch.sqrt(torch.clamp(R ** 2 - dx ** 2, min=0.0))
	return val


def ridge_base_height() -> float:
	x_left = xc - sv
	return float(roof_arc_y(torch.tensor([x_left]))[0].item())


RIDGE_BASE_Y = ridge_base_height()


def roof_y(x: torch.Tensor) -> torch.Tensor:
	"""Piecewise roof profile: arc for |x-xc|>=sv, straight lines to apex otherwise."""
	x = x.clone()
	dy = torch.zeros_like(x)
	mask_arc_left = (x <= xc - sv)
	mask_arc_right = (x >= xc + sv)
	mask_ridge = (~mask_arc_left) & (~mask_arc_right)
	# arc regions
	dy[mask_arc_left] = roof_arc_y(x[mask_arc_left])
	dy[mask_arc_right] = roof_arc_y(x[mask_arc_right])
	# ridge lines
	if mask_ridge.any():
		# two straight lines meeting at apex
		x_left = xc - sv
		y_left = RIDGE_BASE_Y
		x_right = xc + sv
		y_right = RIDGE_BASE_Y
		# left facet line eq through (x_left,y_left) and (apex_x,apex_y)
		m_left = (apex_y - y_left) / (apex_x - x_left)
		b_left = y_left - m_left * x_left
		# right facet
		m_right = (apex_y - y_right) / (apex_x - x_right)
		b_right = y_right - m_right * x_right
		left_sel = mask_ridge & (x <= xc)
		right_sel = mask_ridge & (x > xc)
		dy[left_sel] = m_left * x[left_sel] + b_left
		dy[right_sel] = m_right * x[right_sel] + b_right
	return dy


def inside_domain(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
	"""Return boolean mask: points below roof and within rectangle bounds."""
	return (x >= 0.0) & (x <= W) & (y >= 0.0) & (y <= roof_y(x))



In [ ]:
# ---------------------------
# Data loading and processing
# ---------------------------

def load_day_data(csv_path: str) -> pd.DataFrame:
	"""Load the day data CSV with ambient temperature and C3 measurements.
	Data is collected hourly (every 3600 seconds) over 23 hours."""
	if not os.path.exists(csv_path):
		print(f"Warning: Day data file not found at {csv_path}")
		return pd.DataFrame(columns=['x', 'y1', 't', 'C3', 'Amb_Temp'])

	df = pd.read_csv(csv_path)
	# ensure numeric
	df['t'] = pd.to_numeric(df['t'], errors='coerce')
	df['C3'] = pd.to_numeric(df['C3'], errors='coerce')
	df['Amb_Temp'] = pd.to_numeric(df['Amb_Temp'], errors='coerce')
	df = df.dropna()

	print(f"Loaded {len(df)} hourly data points from day data file")
	print(f"Data spans from {df['t'].min()/3600:.1f}h to {df['t'].max()/3600:.1f}h")
	return df


# Load day data
DAY_DATA_DF = load_day_data(DAY_DATA_PATH)

# Probe location (point C3)
PROBE_X = 3.25
PROBE_Y = 1.2

Loaded 168 hourly data points from day data file
Data spans from 1.0h to 168.0h


In [ ]:
# ---------------------------
# Ambient temperature from data
# ---------------------------

def ambient_temperature_from_data(t_seconds: torch.Tensor) -> torch.Tensor:
	"""Interpolate ambient temperature from day data."""
	if DAY_DATA_DF.empty:
		# Fallback to simple diurnal pattern
		phase = 2 * math.pi * (t_seconds / Tc) - math.pi / 3
		return 8.0 + 6.0 * torch.sin(phase)  # Winter pattern

	# Convert to numpy for interpolation
	t_np = t_seconds.detach().cpu().numpy()
	amb_temp_np = np.interp(t_np, DAY_DATA_DF['t'].values, DAY_DATA_DF['Amb_Temp'].values)
	return torch.tensor(amb_temp_np, dtype=DTYPE, device=t_seconds.device)


In [ ]:
# ---------------------------
# Greenhouse control logic
# ---------------------------

def get_inside_temperature_at_probe(t_seconds: torch.Tensor) -> torch.Tensor:
	"""Get the predicted inside temperature at the probe location."""
	x = torch.full_like(t_seconds, PROBE_X, dtype=DTYPE)
	y = torch.full_like(t_seconds, PROBE_Y, dtype=DTYPE)
	inp = torch.stack([x / Lc, y / Hc, t_seconds / Tc], dim=1).to(DEVICE)
	with torch.no_grad():
		T_hat = model(inp)
	return Theta_c * T_hat[:, 0]


def side_vents_open(t_seconds: torch.Tensor) -> torch.Tensor:
	"""Return 1 if side vents should be open based on inside temperature, else 0."""
	inside_temp = get_inside_temperature_at_probe(t_seconds)
	return (inside_temp >= SIDE_VENT_TEMP).to(t_seconds.dtype)


def roof_vents_open(t_seconds: torch.Tensor) -> torch.Tensor:
	"""Return 1 if roof vents should be open based on inside temperature, else 0."""
	inside_temp = get_inside_temperature_at_probe(t_seconds)
	return (inside_temp >= ROOF_VENT_TEMP).to(t_seconds.dtype)


def heating_active(t_seconds: torch.Tensor) -> torch.Tensor:
	"""Return 1 if heating should be active based on inside temperature, else 0."""
	inside_temp = get_inside_temperature_at_probe(t_seconds)
	return (inside_temp <= HEATING_TEMP).to(t_seconds.dtype)


In [ ]:
# ---------------------------
# Network
# ---------------------------
class MLP(nn.Module):
	def __init__(self, in_dim: int, out_dim: int, hidden: int, layers: int):
		super().__init__()
		mods = []
		mods.append(nn.Linear(in_dim, hidden))
		mods.append(nn.Tanh())
		for _ in range(layers - 2):
			mods.append(nn.Linear(hidden, hidden))
			mods.append(nn.Tanh())
		mods.append(nn.Linear(hidden, out_dim))
		self.net = nn.Sequential(*mods)

	def forward(self, x):
		return self.net(x)


# Model maps (x_hat,y_hat,t_hat) -> (T_hat)
model = MLP(3, 1, HID, LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


In [ ]:

# ---------------------------
# Autograd helpers
# ---------------------------

def gradients(y, x):
	return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True, retain_graph=True, only_inputs=True)[0]



In [ ]:
# ---------------------------
# Samplers
# ---------------------------

def sample_interior(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
	# rejection sampling under roof
	xs = torch.rand(n * 2, dtype=DTYPE) * W
	ys = torch.rand(n * 2, dtype=DTYPE) * H
	mask = inside_domain(xs, ys)
	xs = xs[mask][:n]
	ys = ys[mask][:n]
	if xs.shape[0] < n:
		# pad by recursion
		x2, y2, _ = sample_interior(n - xs.shape[0])
		xs = torch.cat([xs, x2])
		ys = torch.cat([ys, y2])
	ts = torch.rand(n, dtype=DTYPE) * T_final
	return xs, ys, ts


def sample_ground(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
	x = torch.rand(n, dtype=DTYPE) * W
	y = torch.zeros(n, dtype=DTYPE)
	t = torch.rand(n, dtype=DTYPE) * T_final
	return x, y, t


def sample_side(n: int, side: str) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
	"""Sample points on left or right wall, label vent mask for open segment y in [0.3,1.2].
	Bias 70% of samples to the vent span to strengthen BCs."""
	k_vent = int(0.7 * n)
	k_rest = n - k_vent
	if side == 'left':
		xv = torch.zeros(k_vent, dtype=DTYPE)
		xr = torch.zeros(k_rest, dtype=DTYPE)
	elif side == 'right':
		xv = torch.full((k_vent,), W, dtype=DTYPE)
		xr = torch.full((k_rest,), W, dtype=DTYPE)
	else:
		raise ValueError('side must be left or right')
	yv = 0.3 + torch.rand(k_vent, dtype=DTYPE) * (1.2 - 0.3)
	yr = torch.rand(k_rest, dtype=DTYPE) * 1.9
	x = torch.cat([xv, xr], dim=0)
	y = torch.cat([yv, yr], dim=0)
	t = torch.rand(n, dtype=DTYPE) * T_final
	vent_mask = (y >= 0.3) & (y <= 1.2)
	return x, y, t, vent_mask.to(torch.uint8)


def sample_roof(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
	"""Sample roof boundary points and label those on roof vent segment."""
	x = torch.rand(n, dtype=DTYPE) * W
	y = roof_y(x)
	t = torch.rand(n, dtype=DTYPE) * T_final
	# mark roof-vent points near apex along x within width
	vent_mask = (x >= (xc - roof_vent_half_width)) & (x <= (xc + roof_vent_half_width))
	return x, y, t, vent_mask.to(torch.uint8)


def sample_initial(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
	x, y, _ = sample_interior(n)
	t = torch.zeros_like(x)
	return x, y, t


def sample_data_from_csv(df: pd.DataFrame, max_n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
	"""Sample data points from the day data CSV."""
	if df is None or df.empty:
		return torch.empty(0), torch.empty(0), torch.empty(0), torch.empty(0)

	df2 = df.copy()
	# Use up to max_n rows sampled uniformly
	if len(df2) > max_n:
		df2 = df2.sample(n=max_n, random_state=np.random.randint(0, 1_000_000))

	x = torch.full((len(df2),), PROBE_X, dtype=DTYPE)
	y = torch.full((len(df2),), PROBE_Y, dtype=DTYPE)
	t = torch.tensor(df2['t'].values, dtype=DTYPE)
	c3_temp = torch.tensor(df2['C3'].values, dtype=DTYPE)
	return x, y, t, c3_temp


In [ ]:
# ---------------------------
# Physics residuals with heating source
# ---------------------------

def pde_residuals(x, y, t):
	# Normalize inputs
	xh = x / Lc
	yh = y / Hc
	th = t / Tc
	inp = torch.stack([xh, yh, th], dim=1).to(DEVICE)
	inp.requires_grad_(True)

	# Network outputs (dimensionless temperature)
	T_hat = model(inp)
	T = Theta_c * T_hat

	# chain-rule factors
	dx = 1.0 / Lc
	dy = 1.0 / Hc
	dt = 1.0 / Tc

	# gradients w.r.t normalized inputs
	T_xh = gradients(T, inp)[:, 0:1]
	T_yh = gradients(T, inp)[:, 1:2]
	T_th = gradients(T, inp)[:, 2:3]

	# convert to physical derivatives
	T_x = dx * T_xh
	T_y = dy * T_yh
	T_t = dt * T_th

	# second derivatives
	T_xxh = gradients(T_xh, inp)[:, 0:1]
	T_yyh = gradients(T_yh, inp)[:, 1:2]
	T_xx = (dx ** 2) * T_xxh
	T_yy = (dy ** 2) * T_yyh

	# Heating source term
	heating_mask = heating_active(t).view(-1, 1)
	heating_source = heating_mask * (HEATING_POWER / (1000.0 * 1000.0))  # Convert to appropriate units

	# Heat equation residual with heating source
	r_T = T_t - alpha * (T_xx + T_yy) - heating_source
	return r_T


In [ ]:
# ---------------------------
# Loss terms
# ---------------------------

def loss_pde(n_inner: int):
	x, y, t = sample_interior(n_inner)
	x = x.to(DEVICE); y = y.to(DEVICE); t = t.to(DEVICE)
	r_T = pde_residuals(x, y, t)
	return (r_T ** 2).mean()


def loss_walls_and_vents(n_bc: int):
	loss = 0.0

	# ground insulated: penalize dT/dy at y=0
	xg, yg, tg = sample_ground(n_bc)
	inp_g = torch.stack([xg / Lc, yg / Hc, tg / Tc], dim=1).to(DEVICE)
	inp_g.requires_grad_(True)
	T_hat_g = model(inp_g)
	T_g = Theta_c * T_hat_g
	T_yh_g = gradients(T_g, inp_g)[:, 1:2]
	dy = 1.0 / Hc
	T_y_g = dy * T_yh_g
	loss += (T_y_g ** 2).mean()

	# left and right walls: Dirichlet to ambient on vent spans when open, insulated otherwise
	for side in ['left', 'right']:
		xw, yw, tw, vent_mask = sample_side(n_bc, side)
		inp_w = torch.stack([xw / Lc, yw / Hc, tw / Tc], dim=1).to(DEVICE)
		inp_w.requires_grad_(True)
		T_hat_w = model(inp_w)
		T_w = Theta_c * T_hat_w

		# Use temperature-based control for side vents
		open_mask = (side_vents_open(tw.to(DEVICE)) * vent_mask.to(torch.float32).to(DEVICE)).view(-1, 1)
		T_amb = ambient_temperature_from_data(tw.to(DEVICE)).view(-1, 1)
		loss_open = ((T_w - T_amb) ** 2) * open_mask

		T_xh_w = gradients(T_w, inp_w)[:, 0:1]
		dx = 1.0 / Lc
		T_x_w = dx * T_xh_w
		closed_mask = (1.0 - open_mask)
		loss_closed = (T_x_w ** 2) * closed_mask
		loss += loss_open.mean() + loss_closed.mean()

	# roof: Dirichlet on vent when open, insulated otherwise
	xr, yr, tr, vent_mask_r = sample_roof(n_bc)
	inp_r = torch.stack([xr / Lc, yr / Hc, tr / Tc], dim=1).to(DEVICE)
	inp_r.requires_grad_(True)
	T_hat_r = model(inp_r)
	T_r = Theta_c * T_hat_r

	# Use temperature-based control for roof vents
	open_mask_r = (roof_vents_open(tr.to(DEVICE)) * vent_mask_r.to(torch.float32).to(DEVICE)).view(-1, 1)
	T_amb_r = ambient_temperature_from_data(tr.to(DEVICE)).view(-1, 1)
	loss_roof_open = ((T_r - T_amb_r) ** 2) * open_mask_r

	T_yh_r = gradients(T_r, inp_r)[:, 1:2]
	dy = 1.0 / Hc
	T_y_r = dy * T_yh_r
	loss_roof_closed = (T_y_r ** 2) * (1.0 - open_mask_r)
	loss += loss_roof_open.mean() + loss_roof_closed.mean()

	return loss


def loss_initial(n_init: int):
	x, y, t = sample_initial(n_init)
	inp = torch.stack([x / Lc, y / Hc, t / Tc], dim=1).to(DEVICE)
	T_hat = model(inp)
	T = Theta_c * T_hat
	T0 = ambient_temperature_from_data(torch.zeros_like(t).to(DEVICE)).view(-1, 1)
	return ((T - T0) ** 2).mean()


def loss_data(max_n: int):
	"""Data loss using C3 measurements from day data."""
	if DAY_DATA_DF.empty:
		return torch.tensor(0.0, device=DEVICE)

	x, y, t, c3_measured = sample_data_from_csv(DAY_DATA_DF, max_n)
	if x.shape[0] == 0:
		return torch.tensor(0.0, device=DEVICE)

	x = x.to(DEVICE)
	y = y.to(DEVICE)
	t = t.to(DEVICE)
	c3_measured = c3_measured.to(DEVICE)

	inp = torch.stack([x / Lc, y / Hc, t / Tc], dim=1).to(DEVICE)
	T_hat = model(inp)
	T_predicted = Theta_c * T_hat[:, 0]

	return ((T_predicted - c3_measured) ** 2).mean()


# Weights for combined loss
W_PDE = 1.0
W_BC = 10.0  # strengthen boundary conditions
W_INIT = 1.0
W_DATA = 5.0  # weight for data loss



In [ ]:
# ---------------------------
# Visualization helpers
# ---------------------------

def outline_roof(ax):
	xs = np.linspace(0, W, 400, dtype=np.float64)
	xs_t = torch.tensor(xs, dtype=DTYPE)
	ys = roof_y(xs_t).detach().cpu().numpy()
	ax.plot(xs, ys, 'k-', lw=1.5)
	ax.plot([0, 0], [0, 1.9], 'k--', lw=1)
	ax.plot([W, W], [0, 1.9], 'k--', lw=1)
	# side vents
	ax.plot([0, 0], [0.3, 1.2], 'b', lw=4, alpha=0.5)
	ax.plot([W, W], [0.3, 1.2], 'b', lw=4, alpha=0.5)
	# roof vent span
	ax.axvspan(xc - roof_vent_half_width, xc + roof_vent_half_width, color='b', alpha=0.15)


def predict_field(t_seconds: float, nx: int = 120, ny: int = 80):
	xs = np.linspace(0, W, nx)
	ys = np.linspace(0, H, ny)
	X, Y = np.meshgrid(xs, ys)
	xv = torch.tensor(X.flatten(), dtype=DTYPE)
	yv = torch.tensor(Y.flatten(), dtype=DTYPE)
	mask = inside_domain(xv, yv)
	inp = torch.stack([xv[mask] / Lc, yv[mask] / Hc, torch.full_like(xv[mask], t_seconds) / Tc], dim=1).to(DEVICE)
	with torch.no_grad():
		Th = model(inp)
	T = torch.full_like(xv, float('nan'))
	T[mask] = (Theta_c * Th[:, 0])
	return X, Y, T.view(Y.shape).cpu().numpy()


def plot_field(t_seconds: float, fname: str):
	fig, ax = plt.subplots(1, 1, figsize=(7, 4))
	X, Y, T = predict_field(t_seconds)
	c = ax.contourf(X, Y, T, levels=30, cmap='inferno')
	outline_roof(ax)
	ax.set_aspect('equal')
	ax.set_xlim(0, W)
	ax.set_ylim(0, H)
	ax.set_title(f'T [°C] at t={t_seconds/3600:.2f} h')
	fig.colorbar(c, ax=ax, label='°C')
	fig.tight_layout()
	fig.savefig(fname, dpi=150)
	plt.close(fig)


def plot_comparison_timeseries(fname: str):
	"""Plot comparison between predicted and measured data at point C3."""
	# Use the actual data time points for plotting to match measurement intervals
	times = DAY_DATA_DF['t'].values  # Use actual measurement times
	x = torch.full((len(times),), PROBE_X, dtype=DTYPE)
	y = torch.full((len(times),), PROBE_Y, dtype=DTYPE)
	t = torch.tensor(times, dtype=DTYPE)
	inp = torch.stack([x / Lc, y / Hc, t / Tc], dim=1).to(DEVICE)

	with torch.no_grad():
		Th = model(inp)
	predT = (Theta_c * Th[:, 0]).cpu().numpy()
	amb = ambient_temperature_from_data(torch.tensor(times, dtype=DTYPE)).cpu().numpy()

	# Get measured data for comparison
	measured_times = DAY_DATA_DF['t'].values
	measured_c3 = DAY_DATA_DF['C3'].values
	measured_amb = DAY_DATA_DF['Amb_Temp'].values

	fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

	# Plot 1: Temperature comparison
	ax1.plot(times / 3600.0, predT, 'b-', label='PINN Predicted at C3', linewidth=2)
	ax1.plot(measured_times / 3600.0, measured_c3, 'ro', label='Measured C3', markersize=6)
	ax1.plot(times / 3600.0, amb, 'k--', lw=1, alpha=0.7, label='Ambient Temperature')
	ax1.set_xlabel('Time (hours)')
	ax1.set_ylabel('Temperature (°C)')
	ax1.set_title('Hourly Temperature Comparison at Point C3 (x=3.25, y=1.2)')
	ax1.legend()
	ax1.grid(True, ls=':')
	ax1.set_xlim(0, T_final/3600.0)

	# Plot 2: Control system status
	side_vent_status = side_vents_open(torch.tensor(times, dtype=DTYPE)).cpu().numpy()
	roof_vent_status = roof_vents_open(torch.tensor(times, dtype=DTYPE)).cpu().numpy()
	heating_status = heating_active(torch.tensor(times, dtype=DTYPE)).cpu().numpy()

	ax2.plot(times / 3600.0, side_vent_status, 'g-', label='Side Vents Open', linewidth=2)
	ax2.plot(times / 3600.0, roof_vent_status, 'b-', label='Roof Vents Open', linewidth=2)
	ax2.plot(times / 3600.0, heating_status, 'r-', label='Heating Active', linewidth=2)
	ax2.set_xlabel('Time (hours)')
	ax2.set_ylabel('Status (0/1)')
	ax2.set_title('Hourly Greenhouse Control System Status')
	ax2.legend()
	ax2.grid(True, ls=':')
	ax2.set_ylim(-0.1, 1.1)
	ax2.set_xlim(0, T_final/3600.0)

	fig.tight_layout()
	fig.savefig(fname, dpi=150)
	plt.close(fig)


def print_comparison_results():
	"""Print detailed comparison results between predicted and measured data."""
	if DAY_DATA_DF.empty:
		print("No day data available for comparison")
		return

	print("\n" + "="*60)
	print("COMPARISON RESULTS - Point C3 (x=3.25, y=1.2)")
	print("="*60)

	# Get predictions at measured time points
	measured_times = torch.tensor(DAY_DATA_DF['t'].values, dtype=DTYPE)
	x = torch.full_like(measured_times, PROBE_X, dtype=DTYPE)
	y = torch.full_like(measured_times, PROBE_Y, dtype=DTYPE)
	inp = torch.stack([x / Lc, y / Hc, measured_times / Tc], dim=1).to(DEVICE)

	with torch.no_grad():
		Th = model(inp)
	predicted_temps = (Theta_c * Th[:, 0]).cpu().numpy()

	measured_temps = DAY_DATA_DF['C3'].values
	ambient_temps = DAY_DATA_DF['Amb_Temp'].values

	# Calculate statistics
	mae = np.mean(np.abs(predicted_temps - measured_temps))
	rmse = np.sqrt(np.mean((predicted_temps - measured_temps)**2))
	max_error = np.max(np.abs(predicted_temps - measured_temps))

	print(f"Mean Absolute Error (MAE): {mae:.2f} °C")
	print(f"Root Mean Square Error (RMSE): {rmse:.2f} °C")
	print(f"Maximum Error: {max_error:.2f} °C")
	print(f"Number of data points: {len(measured_temps)}")

	print("\nDetailed Hourly Comparison:")
	print("Hour | Predicted(°C) | Measured(°C) | Ambient(°C) | Error(°C)")
	print("-" * 65)

	for i in range(len(measured_times)):
		time_h = measured_times[i].item() / 3600.0
		pred = predicted_temps[i]
		meas = measured_temps[i]
		amb = ambient_temps[i]
		error = pred - meas
		print(f"{time_h:4.1f} | {pred:12.2f} | {meas:12.2f} | {amb:11.2f} | {error:8.2f}")

	# Control system analysis
	print("\n" + "="*60)
	print("CONTROL SYSTEM ANALYSIS")
	print("="*60)

	side_vent_times = []
	roof_vent_times = []
	heating_times = []

	for i, t in enumerate(measured_times):
		inside_temp = predicted_temps[i]
		if inside_temp >= SIDE_VENT_TEMP:
			side_vent_times.append(t.item() / 3600.0)
		if inside_temp >= ROOF_VENT_TEMP:
			roof_vent_times.append(t.item() / 3600.0)
		if inside_temp <= HEATING_TEMP:
			heating_times.append(t.item() / 3600.0)

	print(f"Side vents (≥{SIDE_VENT_TEMP}°C) active at hours: {[f'{t:.1f}h' for t in side_vent_times]}")
	print(f"Roof vents (≥{ROOF_VENT_TEMP}°C) active at hours: {[f'{t:.1f}h' for t in roof_vent_times]}")
	print(f"Heating (≤{HEATING_TEMP}°C) active at hours: {[f'{t:.1f}h' for t in heating_times]}")

	print("\n" + "="*60)


def create_hourly_summary():
	"""Create a summary table with hourly data for better analysis."""
	if DAY_DATA_DF.empty:
		print("No day data available for hourly summary")
		return

	print("\n" + "="*80)
	print("HOURLY SUMMARY TABLE")
	print("="*80)

	# Get predictions at measured time points
	measured_times = torch.tensor(DAY_DATA_DF['t'].values, dtype=DTYPE)
	x = torch.full_like(measured_times, PROBE_X, dtype=DTYPE)
	y = torch.full_like(measured_times, PROBE_Y, dtype=DTYPE)
	inp = torch.stack([x / Lc, y / Hc, measured_times / Tc], dim=1).to(DEVICE)

	with torch.no_grad():
		Th = model(inp)
	predicted_temps = (Theta_c * Th[:, 0]).cpu().numpy()

	measured_temps = DAY_DATA_DF['C3'].values
	ambient_temps = DAY_DATA_DF['Amb_Temp'].values

	# Create hourly summary
	print("Hour | Predicted | Measured | Ambient | Error | Side Vent | Roof Vent | Heating")
	print("     |    (°C)   |   (°C)   |  (°C)   | (°C)  |   (Y/N)   |   (Y/N)   | (Y/N)")
	print("-" * 80)

	for i in range(len(measured_times)):
		time_h = measured_times[i].item() / 3600.0
		pred = predicted_temps[i]
		meas = measured_temps[i]
		amb = ambient_temps[i]
		error = pred - meas

		# Control system status
		side_vent = "Y" if pred >= SIDE_VENT_TEMP else "N"
		roof_vent = "Y" if pred >= ROOF_VENT_TEMP else "N"
		heating = "Y" if pred <= HEATING_TEMP else "N"

		print(f"{time_h:4.1f} | {pred:8.2f} | {meas:8.2f} | {amb:7.2f} | {error:5.2f} | {side_vent:8s} | {roof_vent:8s} | {heating:6s}")

	print("\n" + "="*80)



In [ ]:
# ---------------------------
# Training loop
# ---------------------------

def train():
	start_time = time.time()
	for epoch in range(1, EPOCHS + 1):
		optimizer.zero_grad()
		Lpde = loss_pde(BATCH_F_INNER)
		Lbc = loss_walls_and_vents(BATCH_BC)
		Linit = loss_initial(BATCH_INIT)
		Ldata = loss_data(BATCH_DATA)
		total = W_PDE * Lpde + W_BC * Lbc + W_INIT * Linit + W_DATA * Ldata
		total.backward()
		optimizer.step()

		if epoch % 2000 == 0 or epoch == 1:
			print(f"Epoch {epoch:5d} | L: {total.item():.4e} | PDE: {Lpde.item():.4e} | BC: {Lbc.item():.4e} | IC: {Linit.item():.4e} | DATA: {Ldata.item():.4e}")

		if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
			# save checkpoint
			ckpt_path = os.path.join(OUTPUT_DIR, f'ckpt_{epoch}.pt')
			torch.save({'model': model.state_dict(), 'epoch': epoch}, ckpt_path)

			# plots
			plot_comparison_timeseries(os.path.join(OUTPUT_DIR, f'comparison_{epoch}.png'))
			plot_field(10 * 3600.0, os.path.join(OUTPUT_DIR, f'field_10h_{epoch}.png'))
			plot_field(18 * 3600.0, os.path.join(OUTPUT_DIR, f'field_18h_{epoch}.png'))
			plot_field(23 * 3600.0, os.path.join(OUTPUT_DIR, f'field_23h_{epoch}.png'))

			# print results
			print_comparison_results()
			create_hourly_summary()

			# also dump a small json with losses
			with open(os.path.join(OUTPUT_DIR, f'loss_{epoch}.json'), 'w') as f:
				json.dump({'epoch': epoch, 'Lpde': float(Lpde.item()), 'Lbc': float(Lbc.item()), 'Linit': float(Linit.item()), 'Ldata': float(Ldata.item())}, f, indent=2)

	print(f'Training finished in {(time.time()-start_time)/60:.1f} min')


if __name__ == '__main__':
	print("Starting enhanced PINN training with hourly data and control logic...")
	print(f"Day data loaded: {len(DAY_DATA_DF)} data points")
	print(f"Data duration: {T_final/3600:.1f} hours (0 to {T_final/3600:.1f}h)")
	print(f"Data collection interval: Every 1 hour (3600 seconds)")
	print(f"Control parameters:")
	print(f"  - Side vents open at: {SIDE_VENT_TEMP}°C")
	print(f"  - Roof vents open at: {ROOF_VENT_TEMP}°C")
	print(f"  - Heating starts at: {HEATING_TEMP}°C")
	print(f"  - Heating power: {HEATING_POWER} W/m³")
	print(f"Results will be displayed in hourly format")
	train()


Starting enhanced PINN training with hourly data and control logic...
Day data loaded: 168 data points
Data duration: 168.0 hours (0 to 168.0h)
Data collection interval: Every 1 hour (3600 seconds)
Control parameters:
  - Side vents open at: 23.0°C
  - Roof vents open at: 21.0°C
  - Heating starts at: 8.0°C
  - Heating power: 50.0 W/m³
Results will be displayed in hourly format
Epoch     1 | L: 9.9158e+02 | PDE: 2.5175e-09 | BC: 1.0814e-03 | IC: 9.4092e+00 | DATA: 1.9643e+02
Epoch  2000 | L: 1.0132e+02 | PDE: 1.6910e-07 | BC: 2.1553e-01 | IC: 2.9398e-02 | DATA: 1.9826e+01

COMPARISON RESULTS - Point C3 (x=3.25, y=1.2)
Mean Absolute Error (MAE): 3.33 °C
Root Mean Square Error (RMSE): 4.45 °C
Maximum Error: 8.41 °C
Number of data points: 168

Detailed Hourly Comparison:
Hour | Predicted(°C) | Measured(°C) | Ambient(°C) | Error(°C)
-----------------------------------------------------------------
 1.0 |         3.44 |         7.80 |       -1.49 |    -4.36
 2.0 |         6.74 |         8.1